# Circuit distance of Clifford stages

Question 1 of the paper reads the code a program is in at every cut, with its distance. That is a property of the codes. The **circuit distance** of some stages is a property of the circuit: the smallest number of faults, under a circuit-level noise model, that change the logical information carried through them while no detector notices. A bad schedule of gates can make it smaller than the distance of every code along the way.

`circuit_distance` computes it for Clifford stages with Stim and Mark Webster's `codedistance` library. It writes the stages as a Stim circuit between noiseless checks of the code before and after them (their stabilizers and one logical operator, measured as products of Paulis), with noise after every operation. The detectors and the observable are not written by hand: the tool derives them from the noiseless circuit, as everywhere else in the library, and Stim checks that they are deterministic. `codedistance.circuitDistance` then finds the lightest fault set that flips the observable without firing a detector, by default with its exact integer program `MIPDist`.

This notebook needs the optional packages: `pip install zxqec[distance]` (Stim and `codedistance`).

In [1]:
import time
import matplotlib
matplotlib.use('Agg')                                # the fold-H script chooses an interactive backend on import
import stim
from zxqec.qprogram import (program_from_stim, circuit_distance, z_distance_through_t, without_gates,
                            compare_programs, NotCliffordError, NotPhaseFreeError)
import interactive_fold_h_3d as fth

fold_h = fth.build_program(miswired=False)
for s in range(fold_h.num_stages):
    print("after stage {} ({:22}): code {}".format(s, str(fold_h.stage(s)), fold_h.code_at(s).data().parameters))

after stage 0 (encode + X checks     ): code (9, 1, 3)
after stage 1 (transversal H         ): code (9, 1, 3)
after stage 2 (Z checks (code 2)     ): code (9, 1, 3)
after stage 3 (X checks (code 2)     ): code (9, 1, 3)
after stage 4 (rotate 90°            ): code (9, 1, 3)
after stage 5 (Z checks              ): code (9, 1, 3)
after stage 6 (X checks              ): code (9, 1, 3)
after stage 7 (Z checks + decode     ): code (1, 1, 1)


## The fold-transversal Hadamard

The case study of the paper: the distance-3 surface code, a transversal H, rounds of checks in the rotated code, a 90° rotation back, more rounds, and a decoding onto one qubit at the end. Every code from stage 0 to stage 6 has distance 3. The last stage decodes the logical qubit onto a single physical one, where one fault is always a logical error, so the interesting stages are 1 to 6: from the encoded code to the encoded code.

Noise `kind='full'`: depolarizing noise after every gate, a flip after every reset and before every measurement. The transversal H maps the logical X to the logical Z, so the input logical X is paired with the output logical Z and the other way round; the tool finds which output logical is deterministic by itself.

In [2]:
correct = circuit_distance(fold_h, 1, 6)
print(correct)
print()
print("a lightest fault set that no detector sees:")
for fault in correct.explain():
    print("  ", fault, "  -- stage {} is '{}'".format(fault.stage, fold_h.stage(fault.stage)))

full circuit distance of stages 1-6: 2 (X -> Z: 2, Z -> X: 2)

a lightest fault set that no detector sees:
   stage 3: X15 Y5 after CX 15 5   -- stage 3 is 'X checks (code 2)'
   stage 3: Y16 Y5 after CX 16 5   -- stage 3 is 'X checks (code 2)'


**Circuit distance 2**, although every code on the way has distance 3. Both faults are two-qubit faults right after the CNOT from a check's helper qubit (15 or 16) onto data qubit 5, in the round of X checks after the transversal H. Each puts a Y on qubit 5, and the two cancel. The X on each helper is copied by the check's remaining CNOTs onto the data: onto qubit 6 by helper 15 (`CX 15 6`), onto qubits 7 and 8 by helper 16 (*hook* errors). What is left is X6 X7 X8, a logical X operator of the code, and no measurement result changes (the Z parts that reach helper 16 cancel as well).

The mis-wired version of the thesis's files (one CNOT of a Z check on the wrong qubit; the notebook FoldTransversalHadamard finds it) does worse:

In [3]:
miswired = circuit_distance(fth.build_program(miswired=True), 1, 6)
print(miswired)
for fault in miswired.explain():
    print("  ", fault, "  -- stage {} is '{}'".format(fault.stage, fold_h.stage(fault.stage)))

full circuit distance of stages 1-6: 1 (X -> Z: 2, Z -> X: 1)
   stage 6: Y10 Z1 after CX 10 1   -- stage 6 is 'X checks'


One fault is enough. It is on helper 10 of the X check measured right after the mis-wired Z check, whose result the mis-wired check makes random; in the correct circuit a detector on that result sees the same fault.

### Another CNOT order

Every weight-4 check of the thesis's files runs its CNOTs top-left, top-right, bottom-left, bottom-right. A fault on the helper after the second CNOT spreads onto the last two data qubits, a horizontal pair. In code 2 (after the transversal H) the logical X is horizontal, so the hooks of its X checks (stage 3) can line up into a logical X, as above; in code 1 after the rotation the logical Z is horizontal, so the hooks of the Z checks of stage 5 can line up into a logical Z. The files `circuits/ft_h/ft_h_3.2_n_order.qasm` and `ft_h_5_n_order.qasm` run the weight-4 checks of these two stages in N order (top-left, bottom-left, top-right, bottom-right), which leaves a vertical pair; `build_program(cnot_order='N')` uses them. All checks of a stage have the same type, so their CNOTs commute and the noiseless program is the same, which `compare_programs` confirms. Each change fixes its own round, and with both the circuit distance is 3, the distance of the codes:

In [4]:
n_order = fth.build_program(cnot_order='N')
print("N order against the thesis order:", compare_programs(fold_h, n_order, 0).verdict())
for name, program in [("thesis order", fold_h), ("N order", n_order)]:
    print("{:12}".format(name), "  ".join("stages {}-{}: {}".format(a, b, circuit_distance(program, a, b).distance)
                                         for a, b in [(1, 3), (5, 6), (1, 6)]))
print()
print("a lightest fault set in N order:")
for fault in circuit_distance(n_order, 1, 6).explain():
    print("  ", fault, "  -- stage {} is '{}'".format(fault.stage, n_order.stage(fault.stage)))

N order against the thesis order: logically equivalent


thesis order stages 1-3: 2  stages 5-6: 2  stages 1-6: 2


N order      stages 1-3: 3  stages 5-6: 3  stages 1-6: 3

a lightest fault set in N order:


   stage 2: X8 X12 after CX 8 12   -- stage 2 is 'Z checks (code 2)'
   stage 2: X9 X12 after CX 9 12   -- stage 2 is 'Z checks (code 2)'
   stage 2: X7 X13 after CX 7 13   -- stage 2 is 'Z checks (code 2)'


Nothing in the Stim circuits is hidden: `circuits` holds them, with the derived `DETECTOR` lines and the observable at the end, ready for any other tool that reads Stim. They start with the input boundary: the helpers 14 to 17, which the program prepared in |+⟩ and uses first in stage 3, and the unused qubit 18 (in |0⟩), then the stabilizers and a logical operator of the code.

In [5]:
text = correct.circuits[('X', 'Z')]
lines = text.splitlines()
print(len(lines), "lines;", correct.detectors[('X', 'Z')], "detectors;", stim.Circuit(text).num_qubits, "qubits")
print("\n".join(lines[:15]))
print("...")
print("\n".join(lines[-3:]))

254 lines; 29 detectors; 19 qubits
MPP X14
MPP X15
MPP X16
MPP X17
MPP Z18
MPP X2*X3
MPP X7*X8
MPP X1*X2*X4*X5
MPP X5*X6*X8*X9
MPP Z1*Z4
MPP Z6*Z9
MPP Z1*Z5*Z7*Z8
MPP Z2*Z3*Z5*Z6
MPP X1*X4*X7
TICK
...
DETECTOR rec[-34] rec[-5]
DETECTOR rec[-31] rec[-2]
OBSERVABLE_INCLUDE(0) rec[-26] rec[-1]


## What the function refuses

Stim simulates Clifford circuits only, so a `T` gate, or a correction other than a Pauli fed forward from outcomes, raises `NotCliffordError`. The Z and X distances count Z faults or X faults only, which is meaningful when the circuit never turns one into the other: CNOTs, resets, measurements and Paulis, with a Hadamard only as part of an X-basis reset or measurement (a *phase-free* circuit). Otherwise `NotPhaseFreeError` is raised.

In [6]:
d3 = program_from_stim(open('circuits/cultivation/d3_cultivation.stim').read())[0]
for what, call in [("d = 3 cultivation, with its T gates", lambda: circuit_distance(d3)),
                   ("the Z distance of the fold-H rounds", lambda: circuit_distance(fold_h, 1, 6, kind='Z'))]:
    try:
        call()
    except (NotCliffordError, NotPhaseFreeError) as e:
        print("{}: {}: {}".format(what, type(e).__name__, e))

d = 3 cultivation, with its T gates: NotCliffordError: stage 1: T*(13) is not a Clifford gate this function supports
the Z distance of the fold-H rounds: NotPhaseFreeError: stage 1: a Hadamard on qubit 1 that is not part of an X-basis reset or measurement mixes X and Z faults


## Cultivation: the Z distance through the T gates

A Z fault commutes with `T` and `T†`. In a phase-free circuit whose only non-Clifford gates are single-qubit `T`-type gates, a Z fault therefore flips the same results as in the same circuit with those gates replaced by identities (`without_gates`). The Z distance of the circuit is then the Z distance of the circuit without them, measured against the logical operators of the code the real circuit ends in (for cultivation, T†|+⟩ in the colour code, and the logical X, which Z faults flip), **provided that**

1. the two circuits have the same detectors, and
2. no Z fault can trigger a fed-forward X correction, which would put an X error in front of later `T` gates.

`z_distance_through_t` checks both (the first on the noiseless circuits with and without the `T` gates, between the same boundary checks; the second by finding which results are measured in the X basis), refuses with `TRemovalError` if one fails, and otherwise removes the `T` gates and calls `circuit_distance`. Both cultivation circuits pass: the d = 3 circuit feeds nothing forward, and the 13 X corrections of the d = 5 growth are conditioned on Z-basis results.

In [7]:
results = {}
for name, path in [('d = 3 (Wan, Zhong, Zapirain)', 'circuits/cultivation/d3_cultivation.stim'),
                   ('d = 5 (SOFT repository)', 'circuits/cultivation/d5_cultivation.stim')]:
    real = program_from_stim(open(path).read())[0]
    results[name] = z_distance_through_t(real)
    print("{}: ends in code {}; {}".format(name, real.code_at(real.num_stages - 1).parameters, results[name]))

d = 3 (Wan, Zhong, Zapirain): ends in code (7, 1, 3); Z circuit distance of stages 0-4: 3 (- -> X: 3)


d = 5 (SOFT repository): ends in code (19, 1, 5); Z circuit distance of stages 0-7: 4 (- -> X: 4)


The same by hand: remove the `T` gates and call `circuit_distance` with the code the real circuit ends in. Asked for the X distance too, it gives a number, but **not** the X distance of the real circuit: an X fault before a `T` turns into a Clifford error, which the circuit without `T` does not have. Chan, Strikis, Sun and Cai (arXiv:2609.17706) show that with real `T` gates, d = 3 and d = 5 cultivation have fault distance 2 and 3.

In [8]:
for name, path in [('d = 3', 'circuits/cultivation/d3_cultivation.stim'), ('d = 5', 'circuits/cultivation/d5_cultivation.stim')]:
    real = program_from_stim(open(path).read())[0]
    by_hand = circuit_distance(without_gates(real), kind=('Z', 'X'), code_out=real.code_at(real.num_stages - 1).data())
    print("{}: Z distance {} (as above); X distance of the circuit without T {} (not the real one)".format(
          name, by_hand['Z'].distance, by_hand['X'].distance))

d = 3: Z distance 3 (as above); X distance of the circuit without T 3 (not the real one)


d = 5: Z distance 4 (as above); X distance of the circuit without T 6 (not the real one)


What the checks refuse, on two small programs with a logical input (qubit 1) and a helper in |+⟩ (qubit 0). In the first, `T T` turns the helper into |+i⟩, so its X measurement is random, while without the `T` gates it is fixed: the circuits have different detectors. In the second, an X correction on the logical qubit is controlled by an X-basis result, which a Z fault can flip, and a `T` follows:

In [9]:
from pyzx.circuit import Circuit
from pyzx.circuit.gates import HAD, T, Measurement
from zxqec.qprogram import QProgram, ParityControlled, TRemovalError

def two_qubit_program(gates):
    p = QProgram(2, bits=1, initial_states={0: '+'})
    c = Circuit(2, bit_amount=1)
    for g in gates:
        c.add_gate(g)
    p.add_circuit(c, name='only stage')
    return p

for gates in ([T(0), T(0), HAD(0), Measurement(0, result_bit=0)],
              [HAD(0), Measurement(0, result_bit=0), ParityControlled('X', 1, ['c[0]']), T(1)]):
    try:
        z_distance_through_t(two_qubit_program(gates))
    except TRemovalError as e:
        print("TRemovalError:", e)

TRemovalError: the stages have 0 independent detectors with their T gates and 1 without them, 1 together (input logical X): removing the T gates changes what the detectors see
TRemovalError: stage 0: the X correction on qubit 1 is controlled by c[0], measured in the X basis, so a Z fault can trigger it, and the X error it then makes does not commute with T


At d = 3 the Z distance is 3. At d = 5 it is **4**, the fault distance the d = 5 protocol is designed to have before its escape stage: with four rounds of checks instead of three it would be 5 (Chan et al., Section 2.3). Where the four faults are, next to the code the real circuit is in after each stage (the stages are those of the published file; the d = 5 double-check measures the logical in a code with 37 qubits and 9 logical qubits half way):

In [10]:
d5 = program_from_stim(open('circuits/cultivation/d5_cultivation.stim').read())[0]
faults = results['d = 5 (SOFT repository)'].explain()
for s in range(d5.num_stages):
    print("after stage {}: code {:13}".format(s, str(d5.code_at(s).parameters)),
          "  ".join(str(f) for f in faults if f.stage == s))

after stage 0: code (7, 1, 3)     
after stage 1: code (12, 3, 2)    


after stage 2: code (19, 1, 3)    


after stage 3: code (19, 1, 5)    stage 3: Z17 after CX 18 17


after stage 4: code (19, 1, 5)    stage 4: Z12 after CX 12 18


after stage 5: code (19, 1, 5)    stage 5: Z6 after CX 7 6


after stage 6: code (37, 9, None) 


after stage 7: code (19, 1, 5)    stage 7: Z0 after CX 1 0


Four single Z faults right after CNOTs: one in each of the three rounds of d = 5 checks (stages 3 to 5; the growth ends in stage 3) and one in the second half of the d = 5 double-check (stage 7). Chan et al. find fault distance 3 for the real-T circuit with all faults, which is consistent with a Z distance of 4.

## An independent check with Stim alone

SOFT's published d = 5 file is the protocol (`d5_cultivation.stim`) followed by its authors' own noiseless final check (`d5_final_check.stim`), with their noise model, their `DETECTOR` lines and their observable. Gidney et al. simulate cultivation with `S` and `S†` in place of `T` and `T†`, which Stim can do. Below, Stim searches the published file itself with that replacement in the protocol; the final check's layer of `T` gates (a logical `T†`, which turns T†|+⟩ into a Y eigenstate) is left out, since with `S` the state is already a Y eigenstate. No zxqec code is involved in this cell.

In [11]:
protocol = open('circuits/cultivation/d5_cultivation.stim').read().splitlines()
final_check = open('circuits/cultivation/d5_final_check.stim').read().splitlines()
swap = {'T': 'S', 'T_DAG': 'S_DAG'}
proxy = [swap[l.split(' ', 1)[0]] + ' ' + l.split(' ', 1)[1] if l.split(' ', 1)[0] in swap else l for l in protocol]
proxy += [l for l in final_check if l.split(' ', 1)[0] not in swap]
qc = stim.Circuit('\n'.join(proxy))
lightest = qc.search_for_undetectable_logical_errors(dont_explore_detection_event_sets_with_size_above=5,
                                                     dont_explore_edges_with_degree_above=9999,
                                                     dont_explore_edges_increasing_symptom_degree=False)
print("published file with T -> S:", qc.num_detectors, "detectors, lightest undetectable logical error:", len(lightest))
print("Stim's graph-like search alone:", len(qc.shortest_graphlike_error()))

published file with T -> S: 107 detectors, lightest undetectable logical error: 4
Stim's graph-like search alone: 5


Stim alone finds 4 as well; its graph-like search reports 5 because the lightest error is not graph-like. `circuit_distance` gives the same 4 on the same proxy circuit read as a program:

In [12]:
real = program_from_stim(open('circuits/cultivation/d5_cultivation.stim').read())[0]
proxy_program = program_from_stim('\n'.join(swap[l.split(' ', 1)[0]] + ' ' + l.split(' ', 1)[1]
                                            if l.split(' ', 1)[0] in swap else l for l in protocol))[0]
t0 = time.perf_counter()
r = circuit_distance(proxy_program, code_out=real.code_at(real.num_stages - 1).data())
print(r, "({:.0f} s)".format(time.perf_counter() - t0))

full circuit distance of stages 0-7: 4 (- -> Y: 4) (68 s)


### Gidney et al.'s own d = 5 circuit

Is the 4 a property of SOFT's copy, or of the protocol? `circuits/cultivation/d5_cultivation_gidney_s.stim` is the d = 5 circuit as Gidney, Shutty and Jones published it, with `S` in place of `T` (their data on Zenodo, doi 10.5281/zenodo.13777072, CC BY 4.0, unchanged below a header; see the folder's README). The cell below compares it with SOFT's circuit (with `T → S`, as above), one class of operations at a time.

In [13]:
gidney = stim.Circuit(open('circuits/cultivation/d5_cultivation_gidney_s.stim').read())

def operations(circuit, names):
    # the operations of these kinds, in order, leaving out Paulis fed forward from results (CX rec[-k] q)
    return [str(op) for op in circuit.flattened()
            if op.name in names and not any(t.is_measurement_record_target for t in op.targets_copy())]

for label, names in [('CNOTs', {'CX'}), ('resets and measurements', {'R', 'RX', 'M', 'MX'}),
                     ('noise', {'DEPOLARIZE1', 'DEPOLARIZE2', 'X_ERROR', 'Z_ERROR'}),
                     ('S and S_DAG', {'S', 'S_DAG'}), ('final check (MPP)', {'MPP'})]:
    a, b = operations(gidney, names), operations(qc, names)
    print("{:24} Gidney {:3}  SOFT {:3}  identical: {}".format(label, len(a), len(b), a == b))
print("Paulis fed forward from results: Gidney", sum(1 for op in gidney.flattened() if op.name in ('CX', 'CZ') and
      any(t.is_measurement_record_target for t in op.targets_copy())),
      " SOFT", sum(1 for op in qc.flattened() if op.name in ('CX', 'CZ') and
      any(t.is_measurement_record_target for t in op.targets_copy())))

CNOTs                    Gidney  70  SOFT  70  identical: True
resets and measurements  Gidney  29  SOFT  29  identical: True
noise                    Gidney 180  SOFT 180  identical: True
S and S_DAG              Gidney   5  SOFT   7  identical: False
final check (MPP)        Gidney   1  SOFT   2  identical: False
Paulis fed forward from results: Gidney 0  SOFT 2


The same CNOTs, resets, measurements and noise, in the same order. The differences are bookkeeping: SOFT applies the Pauli corrections of the growth (two lines `CX rec[-k] q` / `CZ rec[-k] q`), while Gidney et al. track them in their `DETECTOR` lines; the d = 5 double-check uses another pattern of `S` and `S†`; and the final check is one `MPP` instead of two. The fault distance of their file, as published, by Stim's search and by `codedistance`'s exact integer program (about two minutes):

In [14]:
import codedistance, contextlib, io
lightest = gidney.search_for_undetectable_logical_errors(dont_explore_detection_event_sets_with_size_above=5,
                                                         dont_explore_edges_with_degree_above=9999,
                                                         dont_explore_edges_increasing_symptom_degree=False)
print("Gidney et al.'s d = 5 file:", gidney.num_qubits, "qubits,", gidney.num_detectors, "detectors")
print("  Stim's search, lightest undetectable logical error:", len(lightest))
print("  Stim's graph-like search:", len(gidney.shortest_graphlike_error()))
t0 = time.perf_counter()
with contextlib.redirect_stdout(io.StringIO()):                # codedistance prints its progress
    exact = codedistance.circuitDistance(gidney, 'MIPDist', params={'filterCircuit': False})
print("  codedistance, MIPDist (exact):", exact['d'], "({:.0f} s)".format(time.perf_counter() - t0))

Gidney et al.'s d = 5 file: 42 qubits, 107 detectors
  Stim's search, lightest undetectable logical error: 4
  Stim's graph-like search: 5


  codedistance, MIPDist (exact): 4 (103 s)


4 again, so the 4 belongs to the published protocol. It is the fault distance d = 5 cultivation is designed to have before its escape stage: Chan, Strikis, Sun and Cai (arXiv:2609.17706, Section 2.3) note that it would be 5 with four rounds of checks instead of three. The four faults found above, one in each of the three rounds of checks and one in the double-check, fit that.

## Summary

| stages | circuit distance | what it shows |
|---|---|---|
| fold-transversal H, stages 1–6 | 2 | two hook errors in stage 3 leave a logical X; every code has distance 3 |
| the same, mis-wired | 1 | the wrong CNOT lets one fault through |
| the same, N order in stages 3 and 5 | 3 | the hooks run across the logical operators; the same noiseless program |
| d = 3 cultivation, Z faults (`z_distance_through_t`) | 3 | both checks pass |
| d = 5 cultivation, Z faults (`z_distance_through_t`) | 4 | four faults across the d = 5 rounds; the real-T circuit is at most 4 |
| d = 5 cultivation with `T → S` | 4 | the same with Stim alone on SOFT's published file |
| d = 5 as Gidney et al. published it (`S`) | 4 | the same CNOTs and noise; 4 is the designed fault distance before the escape stage |

What this does not give: the distance of a stage is measured between noiseless boundary checks, so detectors that compare across the cut are not used and the number is not the fault distance of the whole program; and for circuits with `T` gates only the Z distance is exact, and only when the checks of `z_distance_through_t` pass.